<!-- FILE MAP | Self-contained Colab trainer: pick models in CELL 1, run all cells — no
     configs/run.yaml edits, no commit/push round-trips. Uses train.py's CLI overrides
     (--model/--seed/--epochs), so the training protocol is identical to the canonical runner.
     Resumable: already-trained models (best.pt + metrics.json on Drive) are restored and
     skipped, and train.py mirrors each new best.pt + results to Drive as it goes. -->

# Train all models (Colab)

**Edit only cell 1** (which models, seed, epochs), then *Runtime → Run all*.

- Models train in the order listed (put the cheap ones first: `unet`/`medsam` fit a T4,
  `sam_lora` = ViT-H wants an L4/A100).
- **Crash-safe:** every new best checkpoint and the final results are mirrored to Drive during
  training. Re-running this notebook skips models that already finished — nothing retrains
  unless you set `SKIP_IF_DONE = False`.
- If a model fails mid-list, fix the issue and re-run from cell 4 — finished models are skipped.

In [ ]:
# 1. Pick what to train — the ONLY cell you should need to edit. [TWEAK]
MODELS = ['unet', 'medsam', 'sam_lora']  # any subset/order of: unet | medsam | sam_lora | sam_b
# sam_b = SAM ViT-B + LoRA: same backbone size as MedSAM but generic SAM weights (isolates the confound)
SEEDS = [42, 43, 44]   # trains every model x seed pair; report mean ± std over these (n_runs in configs/base.yaml)
EPOCHS = None          # None = use configs/run.yaml (50); or set an int to override
SKIP_IF_DONE = True    # skip model/seed pairs whose best.pt + metrics.json already exist (Drive or local)

In [ ]:
# 2. Mount Google Drive (checkpoints + results are mirrored here during training).
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# 3. Environment: pull latest code, install deps, fetch data + every backbone MODELS needs.
import os, sys, hashlib, subprocess, zipfile, shutil
from pathlib import Path

REPO = '/content/msu2026summer_final_project'
if not os.path.exists(REPO):
    !git clone --quiet https://github.com/palism1/msu2026summer_final_project.git {REPO}
%cd {REPO}
!git fetch --quiet origin && git reset --hard origin/main
!find . -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null; true
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/facebookresearch/segment-anything.git
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from src.config import MODEL_CHOICES, load_run_config
bad = [m for m in MODELS if m not in MODEL_CHOICES]
assert not bad, f'Unknown model(s) {bad}; choices: {MODEL_CHOICES}'
cfg = load_run_config('configs/run.yaml')

# --- datasets (PraNet train + test packages) ---
DATA_ROOT = Path('data/polyp'); DATA_ROOT.mkdir(parents=True, exist_ok=True)
def _dl_data(file_id, zip_name):
    import gdown
    target = DATA_ROOT / zip_name.replace('.zip', '')
    if target.exists():
        print(f'{target.name}: already present'); return
    zp = f'/tmp/{zip_name}'
    gdown.download(id=file_id, output=zp, quiet=False, fuzzy=True)
    tmp = Path(f'/tmp/x_{target.name}'); tmp.mkdir(exist_ok=True)
    with zipfile.ZipFile(zp) as zf: zf.extractall(tmp)
    ex = [p for p in tmp.iterdir() if not p.name.startswith('__')]
    if len(ex) == 1 and ex[0].is_dir(): shutil.move(str(ex[0]), str(target))
    else:
        target.mkdir(exist_ok=True)
        for p in ex: shutil.move(str(p), str(target / p.name))
    print(f'Done -> {target}')
_dl_data('1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb', 'TrainDataset.zip')
_dl_data('1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao', 'TestDataset.zip')

# --- backbone checkpoints for every selected model ---
def _md5(path, chunk=8*1024*1024):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()

def _wget(url, out):
    subprocess.run(['wget', '-q', '--show-progress', url, '-O', out], check=True)

_SAM_URLS = {'sam_vit_h_4b8939.pth': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
             'sam_vit_b_01ec64.pth': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'}
for _sam_model, _cfg_key in (('sam_lora', 'sam'), ('sam_b', 'sam_b')):
    if _sam_model in MODELS:
        fname = cfg[_cfg_key]['checkpoint']   # sam_lora -> ViT-H, sam_b -> ViT-B
        if not os.path.exists(fname):
            print(f'Downloading SAM checkpoint {fname}...')
            _wget(_SAM_URLS[fname], fname)
        print(f'SAM checkpoint ready: {fname}')
if 'medsam' in MODELS:
    fname = cfg['medsam']['checkpoint']
    MEDSAM_URL = 'https://zenodo.org/records/10689643/files/medsam_vit_b.pth'
    MEDSAM_MD5 = '3bb6db55bd0c9ca30b61248bca72f8d6'
    if os.path.exists(fname) and (os.path.getsize(fname) < 300_000_000 or _md5(fname) != MEDSAM_MD5):
        os.remove(fname)
    if not os.path.exists(fname):
        print('Downloading MedSAM ViT-B checkpoint from Zenodo (~375 MB)...')
        _wget(MEDSAM_URL, fname)
    assert _md5(fname) == MEDSAM_MD5, 'MedSAM MD5 mismatch'
    print(f'MedSAM checkpoint ready: {fname}')
if 'unet' in MODELS:
    print('U-Net: encoder weights download automatically via segmentation-models-pytorch.')

In [ ]:
# 4. Resume state: restore anything already trained from Drive, decide what still needs training.
from src.config import build_run_plan

plans = {(m, s): build_run_plan(cfg, {'model': m, 'seed': s, 'epochs': EPOCHS})
         for m in MODELS for s in SEEDS}
todo = []
print(f'{"model":<10} {"seed":<6} {"checkpoint":<11} {"metrics":<8} -> action')
for (m, s), plan in plans.items():
    local_ckpt = Path(plan.checkpoint_path)
    drive_ckpt = Path(plan.drive_checkpoint_dir) / 'best.pt'
    local_metrics = Path(plan.local_results_dir) / 'metrics.json'
    drive_metrics = Path(plan.drive_results_dir) / 'metrics.json'
    if drive_ckpt.exists() and not local_ckpt.exists():
        local_ckpt.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(drive_ckpt, local_ckpt)
    if drive_metrics.exists() and not local_metrics.exists():
        shutil.copytree(plan.drive_results_dir, plan.local_results_dir, dirs_exist_ok=True)
    done = local_ckpt.exists() and local_metrics.exists()
    action = 'skip (done)' if done and SKIP_IF_DONE else 'TRAIN'
    if action == 'TRAIN':
        todo.append((m, s))
    print(f'{m:<10} {s:<6} {"yes" if local_ckpt.exists() else "no":<11} '
          f'{"yes" if local_metrics.exists() else "no":<8} -> {action}')
print(f'\nWill train: {todo or "nothing — all model/seed pairs already done"}')

In [ ]:
# 5. Train every pending model x seed pair. Each runs in its own process (frees GPU memory
#    between runs); train.py mirrors new best checkpoints + results to Drive as it goes.
for m, s in todo:
    args = [sys.executable, 'train.py', '--config', 'configs/run.yaml',
            '--model', m, '--seed', str(s)]
    if EPOCHS is not None:
        args += ['--epochs', str(EPOCHS)]
    print('=' * 70); print(f'>>> {m} seed{s}: {" ".join(args)}'); print('=' * 70, flush=True)
    proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    if proc.wait() != 0:
        raise RuntimeError(
            f'{m} seed{s} training failed (exit {proc.returncode}). Later models were NOT started; '
            'fix the issue and re-run from cell 4 — finished model/seed pairs will be skipped.')
    if (Path(plans[(m, s)].drive_checkpoint_dir) / 'best.pt').exists():
        print(f'{m} seed{s}: checkpoint persisted to Drive ✓')
    else:
        print(f'{m} seed{s}: WARNING — checkpoint NOT on Drive; copy checkpoints/ manually '
              'before disconnecting.')
print('\nAll requested model/seed pairs trained.')

In [ ]:
# 6. Summary of everything trained so far (reads each metrics.json).
import json
print(f'{"model":<10} {"seed":<6} {"backbone":<9} {"val Dice":>8} {"trainable":>12} {"ckpt MB":>8} '
      f'{"epochs":>6} {"train min":>9}  device')
print('-' * 86)
for (m, s), plan in plans.items():
    mp = Path(plan.local_results_dir) / 'metrics.json'
    if not mp.exists():
        print(f'{m:<10} {s:<6} (no metrics.json yet)')
        continue
    d = json.loads(mp.read_text())
    t = d['timing']
    print(f"{m:<10} {s:<6} {d['backbone']:<9} {d['best_val_dice']:>8.4f} "
          f"{d['params']['trainable']:>12,} {d['checkpoint_size_mb'] or 0:>8.1f} "
          f"{t['epochs_run']:>6} {t['total_seconds']/60:>9.1f}  {d['device_name']}")
print('\nNext: run notebooks/05_benchmark.ipynb for the full five-model comparison (aggregates '
      'across seeds automatically).')